In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from delta.tables import DeltaTable
from datetime import datetime
import uuid

# =============================================================================
# GENERIC AUDIT LOGGING FUNCTION
# Reusable across Bronze, Silver, and Gold layers
# =============================================================================

def log_audit_ingestion(
    pipeline_name: str,
    layer: str,  # "bronze", "silver", or "gold"
    source_name: str,
    target_table: str,
    source_type: str = "file",
    bronze_table: str = None,
    batch_id: str = None,
    run_id: str = None,
    trigger_type: str = "manual",
    attempt: int = 1,
    run_start_ts = None,
    run_end_ts = None,
    last_status: str = "SUCCESS",
    records_read: int = 0,
    records_written: int = 0,
    error_count: int = 0,
    error_message: str = None,
    watermark_col: str = None,
    last_success_watermark_value: str = None,
    current_run_high_watermark_value: str = None,
    file_checkpoint_path: str = None,
    schema_version_applied: str = None,
    producer_system: str = None,
    ingestion_user: str = None,
    notes: str = None,
    last_load_date = None
):
    """
    Generic audit logging function for Bronze, Silver, and Gold layers.
    
    Parameters:
    -----------
    pipeline_name : str (REQUIRED)
        Name of the pipeline (e.g., "HC_Bronze_Ingestion", "HC_Silver_SCD2")
    layer : str (REQUIRED)
        Layer name: "bronze", "silver", or "gold"
    source_name : str (REQUIRED)
        Name of the source (e.g., "employees", "departments")
    target_table : str (REQUIRED)
        Full table name written to (e.g., "edl_hc_datamart.silver.employees")
    source_type : str
        Type of source: "file", "table", "api", etc.
    bronze_table : str
        Bronze table name (for silver/gold layers referencing bronze)
    batch_id : str
        Unique batch identifier
    run_id : str
        Unique run identifier (auto-generated if not provided)
    trigger_type : str
        How was this triggered: "manual", "scheduled", "event"
    attempt : int
        Attempt number (for retries)
    run_start_ts : timestamp
        Start timestamp (auto-captured if not provided)
    run_end_ts : timestamp
        End timestamp (auto-captured if not provided)
    last_status : str
        Status: "SUCCESS", "FAILED", "RUNNING", "PARTIAL"
    records_read : int
        Number of records read from source
    records_written : int
        Number of records written to target
    error_count : int
        Number of errors encountered
    error_message : str
        Error details if failed
    watermark_col : str
        Column used for incremental watermark
    last_success_watermark_value : str
        Last successful watermark value
    current_run_high_watermark_value : str
        Current run's high watermark value
    file_checkpoint_path : str
        Path to checkpoint file for incremental loads
    schema_version_applied : str
        Schema version applied
    producer_system : str
        Source system name
    ingestion_user : str
        User who triggered the ingestion
    notes : str
        Additional notes
    last_load_date : date
        Load date parameter
    
    Returns:
    --------
    str : run_id of the logged audit record
    
    Example Usage:
    --------------
    # Bronze layer
    log_audit_ingestion(
        pipeline_name="HC_Bronze_Ingestion",
        layer="bronze",
        source_name="employees",
        target_table="edl_hc_datamart.bronze.employees",
        records_read=1500,
        records_written=1500,
        last_load_date="2026-02-16"
    )
    
    # Silver layer
    log_audit_ingestion(
        pipeline_name="HC_Silver_SCD2",
        layer="silver",
        source_name="employees",
        target_table="edl_hc_datamart.silver.employees",
        source_type="table",
        bronze_table="edl_hc_datamart.bronze.employees",
        records_read=1500,
        records_written=3200  # includes historical versions
    )
    """
    
    # Auto-generate run_id if not provided
    if run_id is None:
        run_id = str(uuid.uuid4())
    
    # Auto-capture timestamps if not provided
    current_ts = datetime.now()
    if run_start_ts is None:
        run_start_ts = current_ts
    if run_end_ts is None:
        run_end_ts = current_ts
    
    # Calculate duration in milliseconds
    duration_ms = None
    try:
        if isinstance(run_start_ts, datetime) and isinstance(run_end_ts, datetime):
            duration_ms = int((run_end_ts - run_start_ts).total_seconds() * 1000)
    except:
        pass  # Will be calculated in SQL if timestamps are provided
    
    # Prepare audit record
    audit_data = [
        (
            pipeline_name,
            layer,  # NEW COLUMN
            source_type,
            source_name,
            bronze_table,
            target_table,  # NEW COLUMN
            batch_id,
            run_id,
            trigger_type,
            attempt,
            run_start_ts,
            run_end_ts,
            duration_ms,
            last_status,
            records_read,
            records_written,
            error_count,
            error_message,
            watermark_col,
            last_success_watermark_value,
            current_run_high_watermark_value,
            file_checkpoint_path,
            schema_version_applied,
            producer_system,
            ingestion_user,
            notes,
            last_load_date,
            current_ts,  # created_at
            current_ts   # updated_at
        )
    ]
    
    # Define explicit schema with proper data types
    audit_schema = StructType([
        StructField("pipeline_name", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("source_type", StringType(), True),
        StructField("source_name", StringType(), True),
        StructField("bronze_table", StringType(), True),
        StructField("target_table", StringType(), True),
        StructField("batch_id", StringType(), True),
        StructField("run_id", StringType(), True),
        StructField("trigger_type", StringType(), True),
        StructField("attempt", IntegerType(), True),
        StructField("run_start_ts", TimestampType(), True),
        StructField("run_end_ts", TimestampType(), True),
        StructField("duration_ms", IntegerType(), True),
        StructField("last_status", StringType(), True),
        StructField("records_read", IntegerType(), True),
        StructField("records_written", IntegerType(), True),
        StructField("error_count", IntegerType(), True),
        StructField("error_message", StringType(), True),
        StructField("watermark_col", StringType(), True),
        StructField("last_success_watermark_value", StringType(), True),
        StructField("current_run_high_watermark_value", StringType(), True),
        StructField("file_checkpoint_path", StringType(), True),
        StructField("schema_version_applied", StringType(), True),
        StructField("producer_system", StringType(), True),
        StructField("ingestion_user", StringType(), True),
        StructField("notes", StringType(), True),
        StructField("last_load_date", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True)
    ])
    
    # Create DataFrame
    audit_df = spark.createDataFrame(audit_data, audit_schema)
    
    # Calculate duration_ms in SQL if not already calculated
    if duration_ms is None:
        audit_df = audit_df.withColumn(
            "duration_ms",
            F.when(
                (F.col("run_start_ts").isNotNull()) & (F.col("run_end_ts").isNotNull()),
                (F.unix_timestamp("run_end_ts") - F.unix_timestamp("run_start_ts")) * 1000
            ).otherwise(F.lit(None))
        )
    
    # Write to audit table
    audit_table = "edl_hc_datamart.audit.audit_ingestion"
    
    # Check if table exists
    if spark.catalog.tableExists(audit_table):
        # Use merge to update existing records or insert new ones
        target = DeltaTable.forName(spark, audit_table)
        
        target.alias("tgt").merge(
            audit_df.alias("src"),
            "tgt.pipeline_name = src.pipeline_name AND tgt.source_name = src.source_name AND tgt.run_id = src.run_id"
        ).whenMatchedUpdate(set={
            "layer": "src.layer",
            "source_type": "src.source_type",
            "bronze_table": "src.bronze_table",
            "target_table": "src.target_table",
            "batch_id": "src.batch_id",
            "trigger_type": "src.trigger_type",
            "attempt": "src.attempt",
            "run_start_ts": "src.run_start_ts",
            "run_end_ts": "src.run_end_ts",
            "duration_ms": "src.duration_ms",
            "last_status": "src.last_status",
            "records_read": "src.records_read",
            "records_written": "src.records_written",
            "error_count": "src.error_count",
            "error_message": "src.error_message",
            "watermark_col": "src.watermark_col",
            "last_success_watermark_value": "src.last_success_watermark_value",
            "current_run_high_watermark_value": "src.current_run_high_watermark_value",
            "file_checkpoint_path": "src.file_checkpoint_path",
            "schema_version_applied": "src.schema_version_applied",
            "producer_system": "src.producer_system",
            "ingestion_user": "src.ingestion_user",
            "notes": "src.notes",
            "last_load_date": "src.last_load_date",
            "updated_at": "src.updated_at"
        }).whenNotMatchedInsertAll().execute()
        
        print(f"✓ Audit log updated: {audit_table}")
    else:
        # First time - create the table
        audit_df.write.format("delta").mode("append").saveAsTable(audit_table)
        print(f"✓ Audit log created and logged: {audit_table}")
    
    print(f"Run ID: {run_id} | Status: {last_status} | Records: {records_read} → {records_written}")
    return run_id


# =============================================================================
# HELPER FUNCTION: Context-based logging with try/except wrapper
# =============================================================================

def audit_pipeline_run(layer: str, source_name: str, target_table: str, pipeline_func, **kwargs):
    """
    Wrapper function that executes pipeline logic and logs audit automatically.
    
    Parameters:
    -----------
    layer : str
        "bronze", "silver", or "gold"
    source_name : str
        Source name (e.g., "employees")
    target_table : str
        Target table full name
    pipeline_func : callable
        Function to execute (should return records_read, records_written)
    **kwargs : additional audit parameters
    
    Example:
    --------
    def process_employees():
        # ... processing logic ...
        return 1500, 1500  # records_read, records_written
    
    audit_pipeline_run(
        layer="bronze",
        source_name="employees",
        target_table="edl_hc_datamart.bronze.employees",
        pipeline_func=process_employees,
        pipeline_name="HC_Bronze_Ingestion"
    )
    """
    start_ts = datetime.now()
    status = "RUNNING"
    error_msg = None
    records_read = 0
    records_written = 0
    
    try:
        # Execute the pipeline function
        result = pipeline_func()
        
        # Extract metrics if returned
        if isinstance(result, tuple) and len(result) == 2:
            records_read, records_written = result
        
        status = "SUCCESS"
    
    except Exception as e:
        status = "FAILED"
        error_msg = str(e)
        print(f"✗ Pipeline failed: {error_msg}")
        raise  # Re-raise to not hide the error
    
    finally:
        end_ts = datetime.now()
        
        # Log audit regardless of success/failure
        log_audit_ingestion(
            layer=layer,
            source_name=source_name,
            target_table=target_table,
            run_start_ts=start_ts,
            run_end_ts=end_ts,
            last_status=status,
            records_read=records_read,
            records_written=records_written,
            error_message=error_msg,
            error_count=1 if error_msg else 0,
            **kwargs
        )

print("✓ Audit logging functions loaded successfully!")
print("  - log_audit_ingestion(): Direct logging function")
print("  - audit_pipeline_run(): Wrapper with automatic try/except")

In [0]:
%sql
-- Add new columns to existing audit table
-- Run this once to enhance your audit table schema

-- ALTER TABLE edl_hc_datamart.audit.audit_ingestion 
-- ADD COLUMNS (
--   layer STRING COMMENT 'Pipeline layer: bronze, silver, or gold',
--   target_table STRING COMMENT 'Full name of target table written to'
-- );

-- -- Verify the columns were added
-- DESCRIBE edl_hc_datamart.audit.audit_ingestion;

In [0]:
# =============================================================================
# EXAMPLE: How to integrate audit logging into your Bronze pipeline
# =============================================================================

# This is a template showing how to add audit logging to your existing bronze ingestion
# You would integrate this into Cell 2 of this notebook

"""
INTEGRATION APPROACH 1: Add audit logging after each table write

In your main ingestion loop (Cell 2), after the line:
    df_enriched.write.mode("overwrite").format("delta").saveAsTable(full_table)

Add:
    # Count records for audit
    records_written = df_enriched.count()
    records_read = df.count()  # before enrichment
    
    # Log audit
    log_audit_ingestion(
        pipeline_name="HC_Bronze_Ingestion",
        layer="bronze",
        source_name=source_name,
        target_table=full_table,
        source_type=source_type,
        batch_id=md.get('batch_id'),
        run_id=md.get('run_id'),
        trigger_type="manual",  # or "scheduled" if job-triggered
        records_read=records_read,
        records_written=records_written,
        schema_version_applied=md.get('schema_version'),
        producer_system=md.get('producer_system'),
        ingestion_user=md.get('ingestion_user'),
        last_load_date=LOAD_DATE
    )
"""

# =============================================================================
# INTEGRATION APPROACH 2: Wrapper function (cleaner but requires refactoring)
# =============================================================================

"""
You can wrap your processing logic in the audit_pipeline_run function:

def process_source(source_name, landed_path, full_table, options, fmt):
    '''Process a single source and return metrics'''
    
    # Read data
    reader = spark.read
    for k, v in options.items():
        reader = reader.option(k, v)
    df = reader.format(fmt).load(landed_path)
    records_read = df.count()
    
    # ... all your transformation logic ...
    df_enriched = df.withColumn(...)
    
    # Write
    df_enriched.write.mode("overwrite").format("delta").saveAsTable(full_table)
    records_written = df_enriched.count()
    
    return records_read, records_written

# Then call with audit wrapper:
for m in meta_df.collect():
    # ... setup logic ...
    
    audit_pipeline_run(
        layer="bronze",
        source_name=source_name,
        target_table=full_table,
        pipeline_func=lambda: process_source(source_name, landed_path, full_table, options, fmt),
        pipeline_name="HC_Bronze_Ingestion",
        batch_id=md.get('batch_id'),
        run_id=md.get('run_id')
    )
"""

# =============================================================================
# QUICK TEST: Log a sample audit entry
# COMMENTED OUT - Uncomment to test manually
# =============================================================================

# print("\n" + "="*80)
# print("TESTING AUDIT LOGGING")
# print("="*80)

# Test with sample data
# test_run_id = log_audit_ingestion(
#     pipeline_name="HC_Bronze_Ingestion",
#     layer="bronze",
#     source_name="employees_test",
#     target_table="edl_hc_datamart.bronze.employees",
#     source_type="file",
#     trigger_type="manual",
#     records_read=1500,
#     records_written=1500,
#     last_status="SUCCESS",
#     last_load_date="2026-02-16",
#     notes="Test audit log entry from generic notebook"
# )

# print(f"\n✓ Test audit entry created with run_id: {test_run_id}")
# print("\nQuery your audit table:")
# print("SELECT * FROM edl_hc_datamart.audit.audit_ingestion WHERE pipeline_name = 'HC_Bronze_Ingestion' ORDER BY created_at DESC LIMIT 5")